# Turn chunks into embeddings

Tutorial 50 made chunks. Here an embedding model counts its real tokens and turns a few real chunks into equal-length lists of numbers. Nothing is searched or ranked yet; a later retrieval tutorial does that.

The first two code cells find QAChat and check its `.venv` environment. Missing packages are repaired with `uv`. If asked, select the printed `.venv` kernel and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Load an embedding model

`auto` selects MLX on Apple Silicon and PyTorch elsewhere. Each backend uses compatible embedding weights from Hugging Face's normal cache.

The chat model used later for answering is independent from this embedding model.


In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ"


In [ ]:
from qachat.embeddings import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=BACKEND,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


## 2. Prepare real repository chunks

The lower steps are repeated so this tutorial can run by itself. It reads the same real QAChat file used in tutorial 50.


In [ ]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat/repository.py"],
)

print(report["selected_files"][0])


In [ ]:
from qachat.sources.sources import extract_source_units, read_repository_files

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Natural units: {len(units)}")


In [ ]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Chunks: {len(chunks)}")


## 3. Compare words with model tokens

A token can be a word, part of a word, punctuation, or a code mark. The model's token limit—not the visible word count—decides how much text fits.


In [ ]:
first_chunk = chunks[0]

print(first_chunk)


In [ ]:
word_count = len(first_chunk.text.split())

print(f"Visible words: {word_count}")


In [ ]:
token_count = embedding_session.count_tokens(first_chunk.text)

print(f"Model tokens: {token_count}")


## 4. Turn chunks into vectors

One chunk becomes one embedding vector. Every vector from the same model has the same length. Its values are learned numeric features, not readable words.


In [ ]:
selected_chunks = chunks
texts = [chunk.text for chunk in selected_chunks]

print(f"Chunks sent: {len(selected_chunks)}")
for i, text in enumerate(texts):
    print(f"\n>> CHUNK {i} << \n\n {text} \n")


In [ ]:
document_vectors = embedding_session.embed_documents(texts)

print(f"Vectors returned: {len(document_vectors)}")
for i, document_vector in enumerate(document_vectors):
    print(f"\n>> VECTOR {i} << \n\n {document_vector} \n")


In [ ]:
dimensions = {len(vector) for vector in document_vectors}

assert len(document_vectors) == len(selected_chunks)
assert len(dimensions) == 1

dimension = dimensions.pop()

assert dimension > 0

print(f"Vector length: {dimension}")


In [ ]:
for chunk, vector in zip(
    selected_chunks,
    document_vectors,
    strict=True,
):
    name = chunk.symbol or "(module context)"
    print(name, vector[:5])


## 5. Turn a question into a vector

This tutorial uses a Qwen3 embedding model. It expects query text in this model-specific form:

`Instruct: <retrieval instruction>`  
`Query: <user question>`

The instruction says what to retrieve and can change with the user intent. The question contains the user text. `EmbeddingSession` adds neither line; the next cell explicitly applies the Qwen3 format.

In [ ]:
from qachat.query_formats import (
    SOFTWARE_REPOSITORY_INSTRUCTION,
    format_qwen3_query,
)

QUESTION = (
    "Which function scans a repository and selects files by extension?"
)

query_text = format_qwen3_query(
    question=QUESTION,
    instruction=SOFTWARE_REPOSITORY_INSTRUCTION,
)

print(query_text)


In [ ]:
question_vector = embedding_session.embed_query(query_text)

assert len(question_vector) == dimension

print(f"Question vector length: {len(question_vector)}")
print(question_vector)


This format belongs to Qwen3 Embedding, not every embedding model. For a model that expects a plain question, use `query_text = QUESTION`. The reusable `EmbeddingSession` remains model-neutral.

## 6. Compare the question with every chunk

Cosine similarity measures how closely two vectors point in the same direction. A score closer to `1` means the meanings are more similar.

This small example compares every chunk directly. A sensible result ranks `scan_repository` first because it directly matches the question. A later retrieval layer will do this efficiently with an index.

In [ ]:
def cosine_similarity(
    first: list[float],
    second: list[float],
) -> float:
    dot_product = sum(
        first_value * second_value
        for first_value, second_value in zip(
            first,
            second,
            strict=True,
        )
    )
    first_length = sum(value**2 for value in first) ** 0.5
    second_length = sum(value**2 for value in second) ** 0.5
    return dot_product / (first_length * second_length)


ranked_chunks = sorted(
    (
        (cosine_similarity(question_vector, vector), chunk)
        for chunk, vector in zip(
            selected_chunks,
            document_vectors,
            strict=True,
        )
    ),
    key=lambda result: result[0],
    reverse=True,
)

for score, chunk in ranked_chunks:
    name = chunk.symbol or "(module context)"
    print(f"{score:.3f} | {name}")

best_score, best_chunk = ranked_chunks[0]
best_name = best_chunk.symbol or "(module context)"

print("\nMost relevant chunk:")
print(f"{best_score:.3f} | {best_name}")
print(best_chunk.text)


## 7. Release the model


In [ ]:
embedding_session.close()

print("Embedding model released.")


## Result

The chunks and question became vectors in the same numeric space. Cosine similarity then ranked every chunk, and the best match was printed with its source text. This direct comparison is the basic idea behind semantic retrieval; a later tutorial builds a reusable index.